In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
y_col = "Cover_Type"

## 1. データの読み込み

In [ ]:
df_data = pd.read_csv("../data/data.csv")

In [ ]:
df_data.shape

In [ ]:
df_data.head(3)

## 2. データセット作成

In [ ]:
# 欠損値などがないため、前処理はおこわなずに分割する
# データ量が一定量以上あるので、学習:検証:テスト=6:2:2にする

X = df_data.drop(y_col, axis=1).copy()
y = df_data[y_col].copy()

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)

# 検証データを全体に対する割合が 0.8 × 0.25 = 0.2（20%）にする

## 3. 学習と評価

In [ ]:
# 多クラス分類のインスタンス作成

model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=df_data[y_col].nunique(),
    class_weight="balanced",
    random_state=seed,
    verbose=-1,        # ログを非表示
    n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
)

In [ ]:
model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# 保留：
# 比較実験で、条件を合わせるように、X_testとy_test保存しておく？
# X_train_valとかも保存しておく？それを読み込んで、ランダムサンプリングとかする？